### TO DO

- Create parts of speech tags

- plot the most frequent unigram (token) for each timeline

- show the most freq unigrams POS

- show the most freq unigrams NER

- Plot bigrams and trigrams for each timeline

In [141]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import ast
import random
import time
from tqdm.notebook import tqdm

# Shekar
from shekar import POSTagger, NER, WordTokenizer

# NLTK
from nltk import ngrams
import nltk
nltk.download('punkt_tab', quiet = True)

print('Done.')

Done.


[nltk_data] Error loading punkt_tab: <urlopen error [Errno 111]
[nltk_data]     Connection refused>


In [142]:
# Plot settings
sns.set_theme(style = 'whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

In [143]:
# Load the data
df = pd.read_csv('../data/preprocessed/all_processed.csv', encoding = 'utf-8-sig')

In [144]:
# Convert string lists to actual lists
for col in ['tokens', 'tokens_persian', 'tokens_stemmed', 'final_tokens_sent', 'final_tokens_freq']:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11174 entries, 0 to 11173
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   message_id         11174 non-null  int64 
 1   cleaned_text       11174 non-null  str   
 2   timeline           11174 non-null  str   
 3   normalized         11174 non-null  str   
 4   tokens             11174 non-null  object
 5   tokens_persian     11174 non-null  object
 6   tokens_stemmed     11174 non-null  object
 7   final_tokens_sent  11174 non-null  object
 8   final_tokens_freq  11174 non-null  object
dtypes: int64(1), object(5), str(3)
memory usage: 5.9+ MB


In [145]:
df.head(2)

,message_id,cleaned_text,timeline,normalized,tokens,tokens_persian,tokens_stemmed,final_tokens_sent,final_tokens_freq
0,129507,تنها کسی که تو مملکت داره کارش رو خوب انجام می...,timeline_1,تنها کسی که تو مملکت داره کارش رو خوب انجام می...,"[تنها, کسی, که, تو, مملکت, داره, کارش, رو, خوب...","[تنها, کسی, که, تو, مملکت, داره, کارش, رو, خوب...","[تنها, کسی, که, تو, مملک, داره, کار, را, خوب, ...","[کسی, مملک, خوب, انجا, چسب, زن, الویه, نامینو,...","[مملک, انجا, چسب, زن, الویه, نامینو, دادا, یکم..."
1,129512,اولین سالی که بابانوئل برای بچه‌های ما چیزی آو...,timeline_1,اولین سالی که بابانوئل برای بچه‌های ما چیزی آو...,"[اولین, سالی, که, بابانوئل, برای, بچه‌های, ما,...","[اولین, سالی, که, بابانوئل, برای, بچه‌های, ما,...","[اولین, سال, که, بابانوئل, برا, بچه, ما, چیز, ...","[اولین, بابانوئل, آورد, فکر, واکن, دختر, صبح, ...","[بابانوئل, آورد, فکر, واکن, دختر, صبح, بیدار, ..."


In [146]:
# Find top 10 unigrams
TOP_WORDS = {}
TOP_N = 10

for tl in sorted(df['timeline'].unique()):
    tdf = df[df['timeline'] == tl]
    all_tokens = [tok for tokens in tdf['final_tokens_freq'] for tok in tokens]
    cnt = Counter(all_tokens)
    top = cnt.most_common(TOP_N)
    TOP_WORDS[tl] = [word for word, freq in top]

    print(f'\n{tl} top {TOP_N} words:')
    for word, freq in top:
        print(f'    {word}: {freq}')

# Combine all top words among timelines
all_top_words = set()
for words in TOP_WORDS.values():
    all_top_words.update(words)

print(f'\nTotal unique top words across all timelines: {len(all_top_words)}')



timeline_1 top 10 words:
    زندگی: 451
    بابا: 448
    مامان: 436
    خانه: 425
    دختر: 410
    پسر: 390
    فکر: 352
    درس: 344
    مرد: 336
    دست: 317

timeline_2 top 10 words:
    زندگی: 140
    ایران: 109
    آدم: 74
    درس: 64
    مرد: 57
    فکر: 55
    خرید: 46
    جنگ: 45
    خانه: 42
    دست: 42

timeline_3 top 10 words:
    زندگی: 219
    ایران: 188
    اینترنت: 162
    فکر: 137
    مرد: 126
    خانه: 121
    جنگ: 121
    آدم: 120
    بابا: 99
    پسر: 85

Total unique top words across all timelines: 15
